# 6CS012 – Worksheet 5
## End-to-End CNN Model for Image Classification — Fruit in Amazon
**Module:** 6CS012 – Artificial Intelligence and Machine Learning

## Mount Google Drive & Extract Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/AI and Machine Learning/week5/fruitinAmazon.zip'
extract_path = '/content/FruitinAmazon'

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print('Dataset extracted successfully.')
else:
    print('Dataset already extracted.')

train_dir = '/content/FruitinAmazon/train'
test_dir  = '/content/FruitinAmazon/test'

print('Train dir:', train_dir)
print('Test dir :', test_dir)

---
## Task 1: Data Understanding and Visualisation

### 1.1 — Load and Visualise One Sample Image per Class

In [ ]:
import os
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Get the list of class directories from the train folder
class_dirs = sorted([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])
print(f'Classes found ({len(class_dirs)}): {class_dirs}')

In [ ]:
# Select one image randomly from each class
# Display in a grid format with two rows using matplotlib
cols = 3
rows = 2

fig, axes = plt.subplots(rows, cols, figsize=(14, 9))
axes = axes.flatten()

for idx, class_name in enumerate(class_dirs):
    class_path = os.path.join(train_dir, class_name)
    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]
    random_image = random.choice(images)
    img = mpimg.imread(os.path.join(class_path, random_image))
    axes[idx].imshow(img)
    axes[idx].set_title(class_name, fontsize=13, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

**What did you Observe?**

- The dataset contains **6 Amazon fruit classes**: acai, cupuacu, graviola, guarana, pupunha, and tucuma.
- Images vary significantly in **background, lighting, angle, and zoom level**, which is typical of real-world web-scraped datasets.
- Some classes show fruits in clusters (e.g., *pupunha*, *tucuma*), while others show individual fruits (e.g., *graviola*, *cupuacu*) — this variation makes classification more challenging.
- All images are **colour (RGB)**, so the CNN input must handle 3 channels.

### 1.2 — Check for Corrupted Images

In [ ]:
from PIL import Image

corrupted_images = []

for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name)
    if not os.path.isdir(class_path):
        continue
    for image_file in os.listdir(class_path):
        image_path = os.path.join(class_path, image_file)
        try:
            with Image.open(image_path) as img:
                img.verify()
        except (IOError, SyntaxError):
            corrupted_images.append(image_path)
            os.remove(image_path)
            print(f'Removed corrupted image: {image_path}')

if not corrupted_images:
    print('No Corrupted Images Found.')
else:
    print(f'Total corrupted images removed: {len(corrupted_images)}')

---
## Task 2: Loading and Preprocessing Image Data in Keras

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

In [ ]:
img_height       = 128
img_width        = 128
batch_size       = 16
validation_split = 0.2   # 80% training, 20% validation

In [ ]:
# Create a preprocessing layer for normalisation
rescale = tf.keras.layers.Rescaling(1./255)  # Normalise pixel values to [0, 1]

In [ ]:
# Create training dataset with normalisation
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    labels           = 'inferred',
    label_mode       = 'int',
    image_size       = (img_height, img_width),
    interpolation    = 'nearest',
    batch_size       = batch_size,
    shuffle          = True,
    validation_split = validation_split,
    subset           = 'training',
    seed             = 123
)
train_ds = train_ds.map(lambda x, y: (rescale(x), y))

In [ ]:
# Create validation dataset with normalisation
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    labels           = 'inferred',
    label_mode       = 'int',
    image_size       = (img_height, img_width),
    interpolation    = 'nearest',
    batch_size       = batch_size,
    shuffle          = False,
    validation_split = validation_split,
    subset           = 'validation',
    seed             = 123
)
val_ds = val_ds.map(lambda x, y: (rescale(x), y))

In [ ]:
# Create test dataset
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    labels        = 'inferred',
    label_mode    = 'int',
    image_size    = (img_height, img_width),
    interpolation = 'nearest',
    batch_size    = batch_size,
    shuffle       = False,
    seed          = 123
)
test_ds = test_ds.map(lambda x, y: (rescale(x), y))

In [ ]:
class_names = train_ds.class_names
num_classes = len(class_names)
print('Class names:', class_names)
print('Num classes:', num_classes)

---
## Task 3: Implement the CNN Architecture

In [ ]:
model = keras.Sequential([
    # Convolutional Block 1
    layers.Input(shape=(img_height, img_width, 3)),
    layers.Conv2D(32, (3, 3), padding='same', strides=(1, 1), activation='relu'),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),

    # Convolutional Block 2
    layers.Conv2D(32, (3, 3), padding='same', strides=(1, 1), activation='relu'),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),

    # Fully Connected Head
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

In [ ]:
model.summary()

---
## Task 4: Compile the Model

In [ ]:
model.compile(
    optimizer = 'adam',
    loss      = 'sparse_categorical_crossentropy',  # labels are integers
    metrics   = ['accuracy']
)

---
## Task 4: Train the Model

In [ ]:
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath       = 'best_fruit_model.h5',
    monitor        = 'val_accuracy',
    save_best_only = True,
    mode           = 'max',
    verbose        = 1
)

early_stop_cb = tf.keras.callbacks.EarlyStopping(
    monitor              = 'val_loss',
    patience             = 20,
    restore_best_weights = True,
    verbose              = 1
)

In [ ]:
history = model.fit(
    train_ds,
    epochs          = 250,
    batch_size      = 16,
    validation_data = val_ds,
    callbacks       = [checkpoint_cb, early_stop_cb]
)

In [ ]:
# Plot training & validation accuracy and loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Task 5: Evaluate the Model

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f'Test accuracy: {test_acc:.4f}')
print(f'Test loss    : {test_loss:.4f}')

---
## Task 6: Save and Load the Model

In [ ]:
model.save('fruit_cnn_model.h5')
print('Model saved to fruit_cnn_model.h5')

In [ ]:
loaded_model = tf.keras.models.load_model('fruit_cnn_model.h5')
print('Model loaded successfully.')

In [ ]:
# Re-evaluate the loaded model on the test set
loaded_loss, loaded_acc = loaded_model.evaluate(test_ds)
print(f'Loaded model test accuracy: {loaded_acc:.4f}')

---
## Task 7: Predictions and Classification Report

In [ ]:
# Collect all true labels and predictions from test set
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    preds = loaded_model.predict(images, verbose=0)
    y_pred_probs.extend(preds)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.argmax(np.array(y_pred_probs), axis=1)

print('Predicted labels:', [class_names[i] for i in y_pred[:5]])
print('Actual labels   :', [class_names[i] for i in y_true[:5]])

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()